# Importation des bibliothèques

In [7]:
import time
from pathlib import Path

import pandas as pd
import requests

# Constantes

In [8]:
# URL de l'API avec le jeu de données de l'inventaire immobilier de l'État
URL_API = "https://www.data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/inventaire-immobilier-de-letat/records"

# Output path for the data
OUTPUT_PATH = "../src/data/raw/donnees_immobilieres.parquet"

# Fonctions

In [9]:
def get_all_real_estate_data(limit=50, max_records=1000, url=URL_API):
    """
    Récupère l'ensemble des enregistrements de l'inventaire immobilier de l'État
    en gérant la pagination et en limitant le nombre total d'enregistrements.

    Args:
        limit (int): Le nombre d'enregistrements par page (par défaut 50).
        max_records (int): Le nombre maximum d'enregistrements à récupérer (par défaut 1000).

    Returns:
        list: Une liste contenant les enregistrements récupérés, jusqu'à la limite spécifiée.
    """
    all_records = []
    offset = 0
    page_number = 1

    url = url

    while True:
        print(f"Récupération de la page {page_number} (offset: {offset})...")

        # Arrêter si le nombre d'enregistrements récupérés dépasse la limite
        if len(all_records) >= max_records:
            print("Limite d'enregistrements atteinte. Arrêt de la récupération.")
            break

        params = {
            "limit": limit,
            "offset": offset,
        }

        try:
            response = requests.get(url, params=params)
            response.raise_for_status()  # Lève une exception pour les erreurs HTTP

            data = response.json()
            records = data.get("results", [])

            # Si la liste des enregistrements est vide, c'est la fin des données
            if not records:
                print("Toutes les pages ont été récupérées.")
                break

            all_records.extend(records)

            # Incrémenter l'offset pour la prochaine page
            offset += limit
            page_number += 1

            # Pause pour ne pas surcharger le serveur de l'API (bonne pratique)
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"Une erreur est survenue lors de la requête : {e}")
            break

    # S'assurer que le nombre final ne dépasse pas la limite
    return all_records[:max_records]

# Application

In [10]:
# Exemple d'utilisation de la fonction pour récupérer les données
all_data = get_all_real_estate_data(limit=50, max_records=1000)

# Afficher quelques informations sur les données récupérées
if all_data:
    print(f"\nRécupération terminée. Nombre total d'enregistrements : {len(all_data)}")

Récupération de la page 1 (offset: 0)...
Récupération de la page 2 (offset: 50)...
Récupération de la page 3 (offset: 100)...
Récupération de la page 4 (offset: 150)...
Récupération de la page 5 (offset: 200)...
Récupération de la page 6 (offset: 250)...
Récupération de la page 7 (offset: 300)...
Récupération de la page 8 (offset: 350)...
Récupération de la page 9 (offset: 400)...
Récupération de la page 10 (offset: 450)...
Récupération de la page 11 (offset: 500)...
Récupération de la page 12 (offset: 550)...
Récupération de la page 13 (offset: 600)...
Récupération de la page 14 (offset: 650)...
Récupération de la page 15 (offset: 700)...
Récupération de la page 16 (offset: 750)...
Récupération de la page 17 (offset: 800)...
Récupération de la page 18 (offset: 850)...
Récupération de la page 19 (offset: 900)...
Récupération de la page 20 (offset: 950)...
Récupération de la page 21 (offset: 1000)...
Limite d'enregistrements atteinte. Arrêt de la récupération.

Récupération terminée. No

In [11]:
# Création d'un DataFrame à partir de la liste de dictionnaires
df = pd.DataFrame(all_data)

# Affichage du DataFrame avant la sauvegarde
df.head()

,region,ue,id,type,fonction,designation_batiment_terrain,dept,adresse,code_postal,ville,pays,ministere,date_inventaire
0,CORSE,a19e7cbea4c2dae3cfca703bda49e466,936465ba5b332c60491f6cba50a76b49,biens du ministère de l'Intérieur,biens du ministère de l'Intérieur,biens du ministère de l'Intérieur,2B,NaN,NaN,NaN,France,"Intérieur, Outre-mer et Collec. terr.",2014-03
1,NORD-PAS DE CALAIS,c1fb47969da8dd1a5a1a93ae33245ced,44b7ec415ee6f810d44fda9c8c585069,biens du ministère de l'Intérieur,biens du ministère de l'Intérieur,biens du ministère de l'Intérieur,62,NaN,NaN,NaN,France,Intérieur - Outre-mer - Collectivités territor...,2015-10
2,ILE DE France,120994,IG00100036786,Espace naturel,Terrain en friche,MIN de Rungis,94,LD LA BUTTE DE CHEVILLY,94260,Fresnes,France,Services du Premier ministre,2014-03
3,RHONE ALPES,152769,IB00100093449,Bâtiment technique,Bâtiment technique,GENDARMERIE DE SAINT GENIS LAVAL,69,92 Avenue MAL FOCH,69000,SAINT-GENIS-LAVAL,France,France Domaine,2013-05
4,BASSE NORMANDIE,126681,IB00100205180,Bâtiment technique,Dépôt d'archives,ARCHIVES JUSTICE CALVADOS (pré-archivage judic...,14,R NICEPHORE NIEPCE,14120,Mondeville,France,Justice,2017-12


In [12]:
# Créer le répertoire de sortie s'il n'existe pas
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

In [13]:
# Enregistrement du DataFrame dans un fichier Parquet
df.to_parquet(OUTPUT_PATH, index=False)